# Feature Engineering & Data Preparation

The highest-leverage work in any ML project. Covers imputation, encoding, transforms, leakage prevention, and the shuffle-label test.

## Leakage — the most common failure mode

Fitting a scaler on the full dataset before splitting **leaks** test statistics into training.
Always fit transforms inside a Pipeline.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split

X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(5, shuffle=True, random_state=42)

# ❌ WRONG: fit scaler before CV (leaks test set stats into training)
from sklearn.preprocessing import StandardScaler as SS
X_scaled_WRONG = SS().fit_transform(X)   # full dataset used!
wrong_score = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_scaled_WRONG, y, cv=cv, scoring="roc_auc"
).mean()

# ✅ CORRECT: scaler inside Pipeline (fit only on training folds)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, random_state=42)),
])
correct_score = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean()

print(f"WRONG  (scaler fit on full data): ROC-AUC = {wrong_score:.4f}  <- optimistically biased")
print(f"CORRECT (scaler inside Pipeline): ROC-AUC = {correct_score:.4f}  <- honest estimate")
print()
print("The difference looks small here but grows with small datasets and many features")

## Shuffle-label leakage guard

In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
rng  = np.random.default_rng(42)
cv   = StratifiedKFold(5, shuffle=True, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, random_state=42)),
])

real_score      = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean()
shuffled_score  = cross_val_score(pipe, X, rng.permutation(y),
                                   cv=cv, scoring="roc_auc").mean()

print(f"Real labels:     ROC-AUC = {real_score:.4f}")
print(f"Shuffled labels: ROC-AUC = {shuffled_score:.4f}  (should collapse to ~0.50)")
print()
if shuffled_score > 0.65:
    print("WARNING: shuffled labels give high AUC → leakage suspected!")
else:
    print("PASS: shuffled labels collapse to chance — no obvious leakage")

## High-cardinality categoricals — target encoding inside CV

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

rng = np.random.default_rng(42)
n   = 2000

# Simulated e-commerce data: 50 product categories, conversion as target
category = rng.choice([f"cat_{i}" for i in range(50)], n)
y = (rng.uniform(0,1,n) < 0.05 + 0.1 * (category == "cat_0")).astype(int)

df = pd.DataFrame({"category": category, "y": y})

# Target encoding INSIDE CV (to avoid leakage)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
df["target_enc"] = np.nan

for tr_idx, val_idx in kf.split(df):
    train_fold = df.iloc[tr_idx]
    mean_map   = train_fold.groupby("category")["y"].mean()
    global_mean = train_fold["y"].mean()
    df.loc[df.index[val_idx], "target_enc"] = (
        df.iloc[val_idx]["category"].map(mean_map).fillna(global_mean)
    )

print("Target encoding (fold means, no leakage):")
print(df.groupby("category")["target_enc"].mean().sort_values(ascending=False).head(10))
print()
print("cat_0 should have highest encoding (0.15) — injected true effect")